# Match two AnnData objects (1:1)

Matches objects from AnnData 1 to AnnData 2 by nearest centroid within each (Barcode, Well).
Creates a merged AnnData containing:
- obs from AD1 + prefixed obs from AD2 + match metadata
- features (X/var) from AD1 and AD2 with user-defined prefixes


In [ ]:
import os
import numpy as np
import pandas as pd
import anndata as ad
from scipy.spatial import cKDTree
from scipy import sparse

from Fcts_Base import load_adata

%load_ext autoreload
%autoreload 2


# User Input

In [ ]:
# Paths to the two AnnData files produced by 1_FeatureExtraction (separately)
path_ad1 = "PATH/TO/ad1.h5ad"  # e.g. cells
path_ad2 = "PATH/TO/ad2.h5ad"  # e.g. nuclei

# Maximum allowed centroid distance for a match (micrometers)
max_distance_um = 25.0

# Feature prefixes (will be applied to var_names before merging)
prefix_ad1 = "cell"
prefix_ad2 = "nuc"

# Also prefix AD2 obs columns when adding them to AD1 obs
prefix_ad2_obs = prefix_ad2

# Required obs columns
barcode_col = "Barcode"
well_col = "Well"
x_col = "centroid_x_micrometer"
y_col = "centroid_y_micrometer"

# Output
out_dir = os.path.dirname(path_ad1)
out_name = "1b_Matched_1to1.h5ad"
out_path = os.path.join(out_dir, out_name)

# If True: create merged output but raise ValueError if any AD1 objects are unmatched
error_on_unmatched = True


# Load

In [ ]:
ad1 = load_adata(path_ad1, load_zarrs=False)
ad2 = load_adata(path_ad2, load_zarrs=False)

required = [barcode_col, well_col, x_col, y_col]
for name, obj in [("AD1", ad1), ("AD2", ad2)]:
    missing = [c for c in required if c not in obj.obs.columns]
    if missing:
        raise ValueError(f"{name} is missing required obs columns: {missing}")

# Ensure numeric coords
ad1.obs[x_col] = pd.to_numeric(ad1.obs[x_col], errors="coerce")
ad1.obs[y_col] = pd.to_numeric(ad1.obs[y_col], errors="coerce")
ad2.obs[x_col] = pd.to_numeric(ad2.obs[x_col], errors="coerce")
ad2.obs[y_col] = pd.to_numeric(ad2.obs[y_col], errors="coerce")

if ad1.obs[[x_col, y_col]].isna().any().any():
    raise ValueError("AD1 contains NaN centroid coordinates.")
if ad2.obs[[x_col, y_col]].isna().any().any():
    raise ValueError("AD2 contains NaN centroid coordinates.")


# 1:1 matching by proximity (within each well)
Greedy assignment: build all candidate pairs within max_distance_um per (Barcode, Well),
sort by distance, then assign each AD1/AD2 object at most once.

In [ ]:
def _prefix_index(idx, prefix):
    return pd.Index([f"{prefix}__{x}" for x in idx], dtype="object")

def _hstack_X(X1, X2):
    if sparse.issparse(X1) or sparse.issparse(X2):
        X1 = sparse.csr_matrix(X1)
        X2 = sparse.csr_matrix(X2)
        return sparse.hstack([X1, X2]).tocsr()
    return np.concatenate([np.asarray(X1), np.asarray(X2)], axis=1)

matches = []  # (ad1_obs_name, ad2_obs_name, dist_um)
unmatched = []

ad2_groups = dict(tuple(ad2.obs.groupby([barcode_col, well_col], sort=False)))

for (bc, well), g1 in ad1.obs.groupby([barcode_col, well_col], sort=False):
    if (bc, well) not in ad2_groups:
        unmatched.extend(g1.index.tolist())
        continue

    g2 = ad2_groups[(bc, well)]

    coords1 = g1[[x_col, y_col]].to_numpy(dtype=float)
    coords2 = g2[[x_col, y_col]].to_numpy(dtype=float)

    tree2 = cKDTree(coords2)

    # enumerate candidate pairs within radius
    pairs = []
    for i, p in enumerate(coords1):
        js = tree2.query_ball_point(p, r=max_distance_um)
        if not js:
            continue
        dists = np.linalg.norm(coords2[js] - p, axis=1)
        for j, d in zip(js, dists):
            pairs.append((float(d), i, j))

    # greedy 1:1 assignment by smallest distance
    pairs.sort(key=lambda t: t[0])
    used_i = set()
    used_j = set()
    local_map = {}

    for d, i, j in pairs:
        if i in used_i or j in used_j:
            continue
        used_i.add(i)
        used_j.add(j)
        local_map[i] = (j, d)

    g1_obs = g1.index.to_list()
    g2_obs = g2.index.to_list()

    for i, ad1_id in enumerate(g1_obs):
        if i not in local_map:
            unmatched.append(ad1_id)
            continue
        j, d = local_map[i]
        matches.append((ad1_id, g2_obs[j], d))

if len(matches) == 0:
    raise ValueError("No matches found at all. Increase max_distance_um or check centroids / wells / barcodes.")

match_df = pd.DataFrame(matches, columns=["ad1_id", "ad2_id", "match_distance_um"]).set_index("ad1_id")

ad1_m = ad1[match_df.index].copy()
ad2_m = ad2[match_df["ad2_id"].values].copy()

# Prefix feature names to avoid collisions, then build merged X/var
ad1_m.var_names = _prefix_index(ad1_m.var_names, prefix_ad1)
ad2_m.var_names = _prefix_index(ad2_m.var_names, prefix_ad2)

X = _hstack_X(ad1_m.X, ad2_m.X)
var = pd.concat([ad1_m.var.copy(), ad2_m.var.copy()], axis=0)

# Merge obs: keep AD1 obs; add prefixed AD2 obs columns
obs = ad1_m.obs.copy()
for col in ad2_m.obs.columns:
    new_col = f"{prefix_ad2_obs}__{col}"
    if new_col in obs.columns:
        raise ValueError(f"Obs column collision after prefixing: {new_col}")
    obs[new_col] = ad2_m.obs[col].values

# Add match metadata
obs["matched_ad2_id"] = match_df["ad2_id"].values
obs["match_distance_um"] = match_df["match_distance_um"].values

ad_merged = ad.AnnData(X=X, obs=obs, var=var)

# Keep uns from AD1 (so downstream steps keep dataset context)
ad_merged.uns = dict(ad1.uns)
ad_merged.uns["matching"] = {
    "path_ad1": path_ad1,
    "path_ad2": path_ad2,
    "max_distance_um": float(max_distance_um),
    "prefix_ad1": prefix_ad1,
    "prefix_ad2": prefix_ad2,
    "prefix_ad2_obs": prefix_ad2_obs,
    "n_ad1_total": int(ad1.n_obs),
    "n_ad2_total": int(ad2.n_obs),
    "n_matched": int(ad_merged.n_obs),
    "n_unmatched_ad1": int(len(unmatched)),
}

unmatched_path = out_path.replace(".h5ad", "_unmatched_ad1.csv")
pd.DataFrame({"ad1_id": unmatched}).to_csv(unmatched_path, index=False)

ad_merged.write_h5ad(out_path, compression="gzip")

print(f"Saved merged AnnData: {out_path}")
print(f"Saved unmatched report: {unmatched_path}")
print(f"Matched {ad_merged.n_obs}/{ad1.n_obs} objects from AD1.")

if error_on_unmatched and len(unmatched) > 0:
    raise ValueError(
        f"{len(unmatched)} AD1 objects had no AD2 match within {max_distance_um} µm. "
        f"Merged output was written anyway; see {unmatched_path}."
    )
